In [6]:
# Fetching Lead Quality Report Data 

In [46]:
import pandas as pd
import numpy as np
from gspread_pandas import Spread, conf
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.set_option('future.no_silent_downcasting', True)

In [47]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1PagWGqwt9WhpaCSvopMydXaPlsLSlZ2FrAdQMfMSK1g/edit?gid=822921427#gid=822921427"


## LEAD QUALITY REPORT (BY DATE)

In [48]:

target_date_str = "07/04/2026"
target_date_dt  = pd.to_datetime(target_date_str, dayfirst=True)

target_bde_sheets = [
    'RAHIB', 'HABIYA', 'SHAMNA', 
    'CHAITHANYA', 'ZAKIYA', 'SAFAN', 'RANJITH',
    'ADWAITHA', 'NEHA', 'AMINA',
    'ARSHAD', "SHIHAD", "RAHIYAD", "AKASH", "NAJIYA", "NIHAD", "AFNAN"
    # "SHYAMJIL", 'ADHIL', "HIBA", 'NAJA', 'AJIN', 'SHABNA', 'FARSANA', 'SUSHANTHIKA', 'NAFI', 'AJIN', 'SHABNA', 'FARSANA', 'BURHANA', 'ARUN',
]

lead_statuses  = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]
languages      = ["MALAYALAM", "HINDI", "ARABIC", "TAMIL", "ENGLISH", "URDU"]
call_statuses  = ["ANSWERED", "NOT ANSWERED", "NOT VALID"]

mapping_cols = {
    'COUNTRY':       'REGION',
    'AGENT':         'AGENT',
    'CUSTOMER PATH': 'CUSTOMER PATH',
    'PRODUCT 1':     'PRODUCT',
    'PHONE NO 1':    'PHONE NO',
    'STATUS':        'STATUS',
    'LANGUAGE':      'LANGUAGE',
    'DATE':          'DATE',
    'CALL STATUS':   'CALL STATUS'
}
# ─────────────────────────────────────────────


# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────

c = conf.get_config(file_name=config_path)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)

        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            # Standardize dates and filter to target date
            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            subset = subset[subset['DATE'].dt.date == target_date_dt.date()]

            if not subset.empty:
                # Keep only LEAD rows
                subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
                subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
                subset = subset[~subset['AGENT'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]
                subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]
                
                # Standardize string columns
                for col in ['PHONE NO', 'CUSTOMER PATH', 'REGION', 'PRODUCT', 'STATUS', 'LANGUAGE', 'CALL STATUS']:
                    subset[col] = subset[col].astype(str).str.strip().str.upper()
                    subset.loc[subset[col].isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN']), col] = 'UNMARKED'

                subset['_SHEET'] = sheet_name
                all_data_list.append(subset)

    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")


# ── 2. PROCESSING ─────────────────────────────────────────────────────────────

if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)

    # Drop rows with UNMARKED region or product
    # master_df = master_df[
    #     (master_df['REGION']  != 'UNMARKED') &
    #     (master_df['PRODUCT'] != 'UNMARKED')
    # ]

    # master_df = master_df.drop_duplicates(subset=['PHONE NO', 'REGION', 'PRODUCT'], keep='first')

    # ─────────────────────────────────────────────────────────────────────────

    total_leads = (master_df.groupby(['REGION', 'PRODUCT']).size().to_frame('Total Landed Leads'))

    # Pivots now also use the deduplicated data — all counts stay consistent
    # status_pivot = master_df.pivot_table(
    #     index=['REGION', 'PRODUCT'], columns='STATUS',    aggfunc='size', fill_value=0
    # )
    # lang_pivot = master_df.pivot_table(
    #     index=['REGION', 'PRODUCT'], columns='LANGUAGE',  aggfunc='size', fill_value=0
    # )
    # call_pivot = master_df.pivot_table(
    #     index=['REGION', 'PRODUCT'], columns='CALL STATUS', aggfunc='size', fill_value=0
    # )

    # STATUS PIVOT
    status_pivot = master_df.pivot_table(
        index=['REGION', 'PRODUCT'],
        columns='STATUS',
        aggfunc='size',
        fill_value=0
    ).reindex(columns=lead_statuses, fill_value=0)


    # LANGUAGE PIVOT
    lang_pivot = master_df.pivot_table(
        index=['REGION', 'PRODUCT'],
        columns='LANGUAGE',
        aggfunc='size',
        fill_value=0
    ).reindex(columns=languages, fill_value=0)


    # CALL STATUS PIVOT
    call_pivot = master_df.pivot_table(
        index=['REGION', 'PRODUCT'],
        columns='CALL STATUS',
        aggfunc='size',
        fill_value=0
    ).reindex(columns=call_statuses, fill_value=0)
    
    status_pivot = status_pivot.add_suffix('_S')
    lang_pivot   = lang_pivot.add_suffix('_L')
    call_pivot   = call_pivot.add_suffix('_C')

    final_report = total_leads.join(
        [status_pivot, lang_pivot, call_pivot], how='left'
    ).fillna(0).astype(int)

    # ── Enforce column order ──────────────────────────────────────────────────
    ordered_cols = (
        ['Total Landed Leads']
        + [s + '_S' for s in lead_statuses]
        + [l + '_L' for l in languages]
        + [c + '_C' for c in call_statuses]
    )
    valid_cols   = [c for c in ordered_cols if c in final_report.columns]
    final_report = final_report.reindex(columns=valid_cols)

    # ── Rename columns for clean headers ─────────────────────────────────────
    new_names = []
    for col in final_report.columns:
        if   col.endswith('_S'): new_names.append(col.replace('_S', ''))
        elif col.endswith('_L'): new_names.append(col.replace('_L', ''))
        elif col.endswith('_C'): new_names.append(col.replace('_C', ''))
        else:                    new_names.append(col)
    final_report.columns = new_names

  

    # Sort by Region and Product
    final_report = final_report.sort_index(level=['REGION', 'PRODUCT'])

    # ── 3. EXPORT ─────────────────────────────────────────────────────────────
    file_path = f"./Output_Data/Lead_Quality_Report_{target_date_dt.strftime('%d-%m-%Y')}.xlsx"
    final_report.to_excel(file_path, merge_cells=True)

    print(f"\nReport for {target_date_dt.date()} generated successfully.")
    print(f"Total unique leads across all regions: {final_report['Total Landed Leads'].sum()}")
    print(f"Saved to: {file_path}")

else:
    print(f"No records found for {target_date_dt.date()}.")


Report for 2026-04-07 generated successfully.
Total unique leads across all regions: 378
Saved to: ./Output_Data/Lead_Quality_Report_07-04-2026.xlsx


In [10]:
# total_leads_by_agent = (
#         master_df.groupby(['AGENT','REGION', 'PRODUCT'])
#         .size()
#         .to_frame('Total product by agent')
# )
# total_leads_by_agent.to_excel('Total product by agent.xlsx')

## ALL LEAD QUALITY REPORT 

In [49]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'

# Multiple source sheets
sheet_urls = [
    # "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427",
    # "https://docs.google.com/spreadsheets/d/1ztfkIIEeL9EmSdLQIGhBGIFbraMLmqEBwc8FP-CA_3c/edit?gid=351364658#gid=351364658",
    "https://docs.google.com/spreadsheets/d/1PagWGqwt9WhpaCSvopMydXaPlsLSlZ2FrAdQMfMSK1g/edit?"
]

# --- 2. PARAMETERS ---
days_back = 2
today = pd.Timestamp.now().normalize()
start_date = today - pd.Timedelta(days=days_back)


target_bde_sheets = [
    # 'HABIYA', 'CHAITHANYA', 'SAFAN', 'AMINA', "RAHIYAD", "NIHAD"
    'ZAKIYA',  
    'ADWAITHA', 'NEHA',  
    'ARSHAD', "SHIHAD",  "AKASH", "NAJIYA", "RANJITH", "AFNAN",
    # "SHYAMJIL", 'ADHIL', "HIBA", 'NAJA', 'AJIN', 'SHABNA', 'FARSANA',
]
# Remove duplicates from list
target_bde_sheets = list(dict.fromkeys(target_bde_sheets))

lead_statuses   = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]
languages       = ["MALAYALAM", "HINDI", "ARABIC", "TAMIL", "ENGLISH", "URDU", "OTHERS"]
call_statuses   = ["ANSWERED", "NOT ANSWERED", "NOT VALID"]

mapping_cols = {
    'COUNTRY':       'REGION',
    'AGENT':         'AGENT',
    'CUSTOMER PATH': 'CUSTOMER PATH',
    'PRODUCT 1':     'PRODUCT',
    'PHONE NO 1':    'PHONE NO',
    'STATUS':        'STATUS',
    'LANGUAGE':      'LANGUAGE',
    'DATE':          'DATE',
    'CALL STATUS':   'CALL STATUS'
}

# --- 3. FETCH DATA FROM ALL SOURCES ---
c = conf.get_config(file_name=config_path)
all_data_list = []

for url in sheet_urls:
    print(f"\n📌 Connecting to: {url[:60]}...")
    try:
        spread = Spread(url, config=c)
        
        for sheet_name in target_bde_sheets:
            try:
                df = spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
                
                if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
                    subset = df[list(mapping_cols.keys())].copy()
                    subset.rename(columns=mapping_cols, inplace=True)
                    
                    # Date filter
                    subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
                    subset = subset[(subset['DATE'] >= start_date) & (subset['DATE'] <= today)]
                    
                    if not subset.empty:
                        subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
                        subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
                        subset = subset[~subset['AGENT'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]
                        subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]
                        
                        for col in ['PHONE NO', 'STATUS', 'LANGUAGE', 'REGION', 'PRODUCT', 'CALL STATUS']:
                            subset[col] = subset[col].astype(str).str.strip().str.upper()
                            subset.loc[subset[col].isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN']), col] = 'UNMARKED'
                        
                        # Track source
                        subset['SOURCE_URL'] = url[:50]
                        all_data_list.append(subset)
                        print(f"  ✅ {sheet_name}: {len(subset)} records")
                        
            except Exception as e:
                print(f"  ⚠️ Sheet '{sheet_name}' not found in this source: {e}")
                
    except Exception as e:
        print(f"❌ Could not connect to source: {e}")


📌 Connecting to: https://docs.google.com/spreadsheets/d/1PagWGqwt9WhpaCSvopMy...
  ✅ ZAKIYA: 86 records
  ✅ ADWAITHA: 39 records
  ✅ NEHA: 97 records
  ✅ SHIHAD: 77 records
  ✅ AKASH: 102 records
  ✅ NAJIYA: 87 records
  ✅ RANJITH: 88 records
  ✅ AFNAN: 87 records


In [ ]:


# --- 4. PROCESSING & PIVOTING ---
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    print(f"\n✅ Total records collected: {len(master_df)}")

    total_leads  = master_df.groupby(['REGION', 'PRODUCT'])['PHONE NO'].count().to_frame('Total Landed Leads')
    status_pivot = master_df.pivot_table(index=['REGION', 'PRODUCT'], columns='STATUS',      aggfunc='size', fill_value=0)
    lang_pivot   = master_df.pivot_table(index=['REGION', 'PRODUCT'], columns='LANGUAGE',    aggfunc='size', fill_value=0)
    call_pivot   = master_df.pivot_table(index=['REGION', 'PRODUCT'], columns='CALL STATUS', aggfunc='size', fill_value=0)

    # total_leads  = master_df.groupby(['AGENT','REGION'])['PHONE NO'].count().to_frame('Total Landed Leads')
    # status_pivot = master_df.pivot_table(index=['AGENT', 'REGION', ], columns='STATUS',      aggfunc='size', fill_value=0)
    # lang_pivot   = master_df.pivot_table(index=['AGENT', 'REGION', ], columns='LANGUAGE',    aggfunc='size', fill_value=0)
    # call_pivot   = master_df.pivot_table(index=['AGENT', 'REGION', ], columns='CALL STATUS', aggfunc='size', fill_value=0)

    # total_leads  = master_df.groupby(['AGENT', 'PRODUCT'])['PHONE NO'].count().to_frame('Total Landed Leads')
    # status_pivot = master_df.pivot_table(index=['AGENT', 'PRODUCT'], columns='STATUS',      aggfunc='size', fill_value=0)
    # lang_pivot   = master_df.pivot_table(index=['AGENT', 'PRODUCT'], columns='LANGUAGE',    aggfunc='size', fill_value=0)
    # call_pivot   = master_df.pivot_table(index=['AGENT', 'PRODUCT'], columns='CALL STATUS', aggfunc='size', fill_value=0)

    # total_leads  = master_df.groupby(['AGENT', 'REGION'])['PHONE NO'].count().to_frame('Total Landed Leads')
    # status_pivot = master_df.pivot_table(index=['AGENT', 'REGION'], columns='STATUS',      aggfunc='size', fill_value=0)
    # lang_pivot   = master_df.pivot_table(index=['AGENT', 'REGION'], columns='LANGUAGE',    aggfunc='size', fill_value=0)
    # call_pivot   = master_df.pivot_table(index=['AGENT', 'REGION'], columns='CALL STATUS', aggfunc='size', fill_value=0)

    # total_leads  = master_df.groupby(['AGENT', 'REGION', 'PRODUCT'])['PHONE NO'].count().to_frame('Total Landed Leads')
    # status_pivot = master_df.pivot_table(index=['AGENT', 'REGION', 'PRODUCT'], columns='STATUS',      aggfunc='size', fill_value=0)
    # lang_pivot   = master_df.pivot_table(index=['AGENT', 'REGION', 'PRODUCT'], columns='LANGUAGE',    aggfunc='size', fill_value=0)
    # call_pivot   = master_df.pivot_table(index=['AGENT', 'REGION', 'PRODUCT'], columns='CALL STATUS', aggfunc='size', fill_value=0)

    status_pivot = status_pivot.add_suffix('_S')
    lang_pivot   = lang_pivot.add_suffix('_L')
    call_pivot   = call_pivot.add_suffix('_C')

    final_report = total_leads.join([status_pivot, lang_pivot, call_pivot], how='left').fillna(0).astype(int)

    ordered_cols = (['Total Landed Leads'] +
                    [s + '_S' for s in lead_statuses] +
                    [l + '_L' for l in languages] +
                    [c + '_C' for c in call_statuses])

    valid_cols   = [c for c in ordered_cols if c in final_report.columns]
    final_report = final_report.reindex(columns=valid_cols)

    new_names = []
    for col in final_report.columns:
        if   col.endswith('_S'): new_names.append(col.replace('_S', ' '))
        elif col.endswith('_L'): new_names.append(col.replace('_L', ' '))
        elif col.endswith('_C'): new_names.append(col.replace('_C', ' '))
        else: new_names.append(col)

    final_report.columns = new_names
    final_report = final_report.sort_index(level=['REGION', 'PRODUCT'])
    # final_report = final_report.sort_index(level=['AGENT','PRODUCT'])
    # final_report = final_report.sort_index(level=['AGENT', 'REGION'])
    # final_report = final_report.sort_index(level=['AGENT','REGION'])
    # final_report = final_report.sort_index(level=['AGENT','REGION','PRODUCT'])

    # --- 5. EXPORT ---
    date_range_str = f"{start_date.strftime('%d%m')}_to_{today.strftime('%d%m')}"
    file_path = f"./Output_Data/Lead_Quality_Report_{date_range_str}.xlsx"
    # file_path = f"./Output_Data/Product_wise_Lead_Quality_Report.xlsx"
    # file_path = f"./Output_Data/Agent_Region_Product_wise_Lead_Quality_Report.xlsx"
    final_report.to_excel(file_path, merge_cells=True)

    print(f"\n📊 Report saved to: {file_path}")
else:
    print("No records found across all sources.")


✅ Total records collected: 663


PermissionError: [Errno 13] Permission denied: './Output_Data/Lead_Quality_Report_0604_to_0804.xlsx'